## Environment Setup

In [13]:
import os
import xpd_tools
from tkinter.constants import N
import pandas as pd
import numpy as np

## AGENT - Initialization

### Initialize the Agent and set the logical paths and globals of our system.

For Bluesky:
- Queueserver toggle
- Tiled profile

Other:
- HTTP API and Server uri's
- ZMQ consumer addresses

Evaluation methods:
- xray, uvvis, or xray-uvis


In [14]:
from xpd_tools.optimization.agent import BuildAgent

# Queueserver connection
HTTP_SERVER_URI = os.environ.get(
    "QSERVER_HTTP_URI", 
    "https://xf28id2-xpd-qs1.nsls2.bnl.gov"
    )
HTTP_API_KEY = os.environ.get(
    "QSERVER_HTTP_SERVER_API_KEY", 
    ""
    )
ZMQ_CONSUMER_ADDR = os.environ.get(
    "ZMQ_CONSUMER_ADDR",
    "ipc:///var/lib/bluesky-zmq-proxy/xpd-ipc-in-ipc-out/out.sock",
    )

# Historical data path
#AGENT_DATA_PATH = "tmp/checkpoints/agent_halide_data.csv"
AGENT_DATA_PATH = None

# Tiled URI for evaluation function
TILED_URI = os.environ.get(
    "TILED_URI", 
    "https://tiled.nsls2.bnl.gov"
    )
TILED_PROFILE = os.environ.get(
    "TILED_PROFILE", 
    "xpd"
    )

# Acquisition plan name (must be registered on the queueserver)
ACQUISITION_PLAN_NAME = "xray_uvvis_acquire"

build_agent = BuildAgent(
    # Use queueserver for async
    queue_server = True,
    # Agent
    agent_data_path = AGENT_DATA_PATH,
    # Http 
    http_server_uri = HTTP_SERVER_URI,
    http_api_key = HTTP_API_KEY,
    # ZMQ
    zmq_consumer_address = ZMQ_CONSUMER_ADDR,
    # Tiled
    tiled_profile = TILED_PROFILE,
    # Evaluation Method
    evaluation_method = 'xray'
)


# Provide the agent some metadata
build_agent.set_metadata({
    "beamline": "28id2",
    "tags": ["qserver", "bluesky"],
    "comment": "Halide Synthesis Test"
    })
print(build_agent.metadata_string)

beamline: 28id2
tags: ['qserver', 'bluesky']
comment: Halide Synthesis Test



## BEAMLINE - PDF Xray Diffraction

Objective correlation function with the phases that will be tested.


Correlation functions:
- pearson, nn-matrix, weighted-profile-r, cross-correlation, or ensemble

In [ ]:
from xpd_tools.optimization.helpers.phases import Phase

# PDF correlation function
PDF_FUNCTION = "pearson"

# Set the phase selection objectives and directions
build_agent.set_xray_objectives(
    # Measurement
    max_retries     = 10,
    retry_delay     = 2.0,
    # Configuration
    exposure        = 600.0,
    frame_acq_time  = 1.5,
    no_dark         = False,
    stream_name     = "scattering",
    # Quality Checks
    use_good_bad    = True,
    good_target     = 2,
    max_bad         = 3,
    num_abs         = 16,
    num_flu         = 16,
    # Fitting Masks
    min_radius      = 2.0,
    max_radius      = 20.0
    # Phase Fitting
    objective_function = PDF_FUNCTION,
    phases   = (
        Phase(
            name      = "CsPbBr3",
            gr        = "refdata/CsPbBr3.gr",
            cif       = "refdata/CsPbBr3.cif",
            minimize  = False
            ),
        Phase(
            name      = "CsBr",
            gr        = "refdata/CsBr.gr",
            cif       = "refdata/CsBr.cif",
            minimize  = True
            ),
        Phase(
            name      = "Cs4PbBr6",
            gr        = "refdata/Cs4PbBr6.gr",
            cif       = "refdata/Cs4PbBr6.cif",
            minimize  = True
            ),
    )
  )
print(build_agent.set_xray_objectives)

## BEAMLINE - UVvis Screening

In [ ]:
# Optimization Parameters
PEAK_TARGET         = 450   # nm
PEAK_TOLERANCE      = 5     # nm

from xpd_tools.optimization.helpers.qepro import PlqyReference

build_agent.set_uvvis_objectives(
    # Objective Target
    peak_target             = PEAK_TARGET,
    peak_tolerance          = PEAK_TOLERANCE,
    max_retries             = 10,
    retry_delay             = 2.0,
    # Screening
    screen_key_height       = 200,
    screen_peak_height      = 50,
    screen_peak_distance    = 100,
    # Processing
    process_percent_range_pl    = (40, 100),
    process_percent_range_abs   = (10, 70),
    process_wavelength_range    = (210, 700),
    # Fitting
    fit_pl_wavelength_range     = (400, 800),
    fit_pl_maxfev               = 100000,
    fit_abs_baseline_maxfev     = 10000,
    fit_r2_window_sigma         = 3,
    # Calibration Standard Reference (reference_type defaults to "quinine")
    plqy = PlqyReference(
        excitation_wavelength_nm   = 365,
        absorbance                 = 0.361,
        pl_integral                = 952628,
        refractive_index           = 1.337,
        plqy                       = 0.546,
        solvent_refractive_index   = 1.506,
    ),
)

print(build_agent.plqy)

## BLOP - DOFs

Degrees of freedom that BLOP can modify

In [ ]:
# Set the Agent DOFs
from xpd_tools.optimization.helpers.dofs import Pump

build_agent.set_dofs(
    pumps = (
      Pump(
        name = "CsPb",
        bounds = [10,200],
        id = "dds2_p1"
      ),
      Pump(
        name = "Br",
        bounds = [5,200],
        id = "dds2_p2"
      ),
      Pump(
        name = "I2",
        bounds = [0,200],
        id = "dds3_p1"
      )
    )
  )
print(build_agent.dofs)

## BEAMLINE - Experimental Harware Parameters

In [18]:
from xpd_tools.optimization.plans import FlowSource, DilutionStage, WashCycle

build_agent.experiment(
    sources = (
        FlowSource(
            dof="infusion_rate_CsPb",
            pump='dds2_p1',
            precursor="CsPbOA",
            sample_label="CsPb",
        ),
        FlowSource(
            dof="infusion_rate_Br",
            pump='dds2_p2',
            precursor="TOABr",
            sample_label="Br",
        ),
        FlowSource(
            dof="infusion_rate_I2",
            pump='dds3_p1',
            precursor="ZnI2",
            sample_label="I2",
        ),
    ),
    dilutions = (
        DilutionStage(
            pump='dds1_p1',
            ratio=1.0,
            position="before_equilibrium",
            syringe_ml=20,
            material="plastic_BD",
            target_ml=20,
        ),
        DilutionStage(
            pump='ultra2',
            ratio=1.0,
            position="after_equilibrium",
            syringe_ml=100,
            material="steel",
            target_ml=100,
            wait_sec=30,
        ),
    ),
    wash_cycles=(
        WashCycle(pump='ultra1'),
    )
)

## BLOP - Load and preprocess historical data

In [19]:
if AGENT_DATA_PATH is not None:
    df = pd.read_csv(AGENT_DATA_PATH, index_col=0)
    df = df[["Peak", "log_FWHM", "log_PLQY", "infusion_rate_CsPb", "infusion_rate_Br", "infusion_rate_Cl"]]
    df["peak_distance"] = (df["Peak"] - PEAK_TARGET).abs()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

## BLOP - Build Agent

In [20]:
run_agent = build_agent.build()
run_agent.ax_client.configure_generation_strategy(
    initialize_with_center=False,
    # Loads historical data?
    use_existing_trials_for_initialization=True,
)

AttributeError: 'NoneType' object has no attribute 'ax_client'

## Summarize and Export Data

In [ ]:
df = run_agent.ax_client.summarize()
df.to_csv("tmp/output/agent_halide_data.csv")